# Segmentação de Clientes — Vértice Investimentos
**Análise de Clustering com K-Means**  
*Ciência de Dados aplicada a finanças*

---
## Entendimento do Problema

### O que o Rafael está pedindo?

O Rafael, Head de Relacionamento da Vértice, tem um problema prático: com 500 carteiras ativas, a equipe trata todos os clientes da mesma forma — mesmo conteúdo, mesmo acompanhamento, mesmas sugestões. Ele suspeita que existem **perfis de comportamento distintos** na base e quer entender quem são esses grupos para personalizar o atendimento.

**Hipótese analítica verificável:**

> Existem grupos naturais de clientes na base da Vértice, diferenciáveis pelo seu comportamento de investimento (alocação, frequência de operações, concentração, prazo de posição e volume), que não são capturados adequadamente pelo perfil de suitability declarado.

Dito de outra forma: os dados comportamentais têm estrutura de clusters que permite segmentar os clientes de forma mais fiel à realidade do que a autodeclaração de perfil de risco.

### Por que isso é aprendizado não supervisionado?

Não temos uma variável-alvo definida previamente. O Rafael não sabe — e ninguém sabe — quantos perfis existem nem quais são eles. O problema não é prever uma categoria conhecida, mas **descobrir estruturas latentes** nos dados. Isso caracteriza um problema de **aprendizado não supervisionado**, especificamente de **clusterização**.

A coluna `perfil_declarado_pelo_cliente` (conservador, moderado, arrojado) parece um rótulo, mas o próprio Rafael levanta a hipótese de que declaração e comportamento real divergem. Por isso, **essa variável será excluída do modelo** — ela será usada apenas depois, para validar e comparar os resultados com o que os clientes disseram sobre si mesmos.

---
## Importações e Carregamento dos Dados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import warnings
warnings.filterwarnings('ignore')


df = pd.read_csv("carteiras_clientes.csv")
print(f"Dataset: {df.shape[0]} clientes e {df.shape[1]} variáveis")
df.head()

---
## Análise Exploratória (EDA)

Antes de modelar, precisamos entender a estrutura dos dados: distribuições, correlações, outliers e possíveis inconsistências. Cada achado aqui vai influenciar as decisões de pré-processamento.

### Visão geral estatística

In [ ]:
# Estatísticas descritivas das variáveis numéricas
num_cols = df.select_dtypes(include='number').drop(columns='id_cliente').columns
df[num_cols].describe().round(2)

**Observações iniciais:**
- `patrimonio_declarado_R$` e `valor_total_investido_R$` têm alta variância (desvio padrão próximo à média), sugerindo grande heterogeneidade e presença de outliers.
- `operacoes_por_mes` vai de 0 a 56 — enorme amplitude.
- `prazo_medio_posicao_dias` vai de 3 a 927 — clientes com comportamentos muito distintos.
- Sem valores nulos em nenhuma coluna.

### Distribuição das variáveis numéricas

In [ ]:
fig, axes = plt.subplots(5, 3, figsize=(16, 18))
axes = axes.flatten()

plot_cols = [c for c in num_cols]
for i, col in enumerate(plot_cols):
    axes[i].hist(df[col], bins=30, color='steelblue', edgecolor='white', alpha=0.85)
    axes[i].set_title(col, fontsize=10, fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].tick_params(labelsize=8)

# Esconder eixos extras
for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Distribuição das Variáveis Numéricas', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()

- `perc_renda_fixa` = há uma concentração de clientes com quase 0% e outra com quase 100% em renda fixa — perfis opostos.
- `perc_acoes` e `perc_crypto` = são assimétricos à direita: maioria aloca pouco, mas há uma cauda de clientes com alta exposição.
- `valor_total_investido_R$` = tem forte assimetria — a maioria investe valores menores, mas há clientes com patrimônio muito elevado.
- `operacoes_por_mes` e `prazo_medio_posicao_dias` = seguem distribuições com caudas longas, indicando grupos com comportamentos operacionais bem distintos.

### Correlação entre as variáveis de alocação

In [ ]:
perc_cols = ['perc_renda_fixa', 'perc_acoes', 'perc_fiis', 'perc_crypto', 'perc_exterior']

corr = df[perc_cols].corr()

fig, ax = plt.subplots(figsize=(7, 5))
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title('Correlação entre Variáveis de Alocação', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

**Achado relevante:**
- `perc_renda_fixa` e `perc_acoes` têm correlação de **-0.90** — fortíssima correlação negativa. Quem aloca mais em ações, aloca menos em renda fixa, e vice-versa. Faz sentido: a carteira soma ~100%.
- `perc_crypto` e `perc_renda_fixa` também têm correlação negativa forte (-0.65): clientes com cripto tendem a ter menos renda fixa.
- `perc_fiis` e `perc_exterior` são relativamente independentes das demais.


### Análise de outliers

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

outlier_cols = ['valor_total_investido_R$', 'operacoes_por_mes', 'prazo_medio_posicao_dias']
titles = ['Valor Investido (R$)', 'Operações por Mês', 'Prazo Médio da Posição (dias)']

for ax, col, title in zip(axes, outlier_cols, titles):
    bp = ax.boxplot(df[col], vert=True, patch_artist=True,
                    boxprops=dict(facecolor='steelblue', alpha=0.6),
                    medianprops=dict(color='darkred', linewidth=2))
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(
        lambda x, _: f'{x/1e6:.1f}M' if col == 'valor_total_investido_R$' and x >= 1e6
        else f'{x/1e3:.0f}k' if col == 'valor_total_investido_R$' else f'{x:.0f}'))

plt.suptitle('Outliers nas Principais Variáveis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

# Clientes acima do percentil 95 em operações
p95_op = df['operacoes_por_mes'].quantile(0.95)
p95_prazo = df['prazo_medio_posicao_dias'].quantile(0.95)
print(f"Clientes com >P95 em operações/mês (>{p95_op:.0f}): {(df['operacoes_por_mes'] > p95_op).sum()}")
print(f"Clientes com >P95 em prazo posição (>{p95_prazo:.0f} dias): {(df['prazo_medio_posicao_dias'] > p95_prazo).sum()}")

**Outliers identificados:**
- Há clientes com `valor_total_investido_R$` muito acima da mediana — alguns poucos "grandes investidores".
- `operacoes_por_mes` tem outliers extremos: alguns clientes operam dezenas de vezes por mês enquanto a mediana é ~3.
- `prazo_medio_posicao_dias` tem casos extremos próximos a 900 dias — clientes que literalmente nunca mexem na carteira.

**Decisão:** não removeremos esses outliers. Eles provavelmente representam perfis reais e distintos (o trader ativo, o grande investidor, o cliente inativo). O que faremos é **normalizar os dados** para que essas variáveis não dominem o modelo pela escala.

### Perfil declarado vs. comportamento observado — primeira olhada

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Operações por mês por perfil declarado
order = ['conservador', 'moderado', 'arrojado']
colors = ['#4C72B0', '#DD8452', '#55A868']

ax = axes[0]
for i, perfil in enumerate(order):
    subset = df[df['perfil_declarado_pelo_cliente'] == perfil]['operacoes_por_mes']
    ax.violinplot(subset, positions=[i], widths=0.6, showmedians=True)
ax.set_xticks([0,1,2])
ax.set_xticklabels(order)
ax.set_title('Operações/mês por Perfil Declarado', fontweight='bold')
ax.set_ylabel('Operações por mês')

# % renda fixa por perfil declarado
ax2 = axes[1]
df.groupby('perfil_declarado_pelo_cliente')['perc_renda_fixa'].mean().reindex(order).plot(
    kind='bar', ax=ax2, color=colors, edgecolor='white', width=0.5)
ax2.set_title('% Médio em Renda Fixa por Perfil Declarado', fontweight='bold')
ax2.set_ylabel('% Renda Fixa (média)')
ax2.set_xlabel('')
ax2.set_xticklabels(order, rotation=0)
ax2.bar_label(ax2.containers[0], fmt='%.1f%%', padding=3)

plt.suptitle('Perfil Declarado vs. Comportamento Real — Primeiros Indícios', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("Estatísticas de operações por perfil declarado:")
print(df.groupby('perfil_declarado_pelo_cliente')['operacoes_por_mes'].describe().round(2))

**Primeiro sinal de alerta:**
- Clientes "arrojados" têm em média mais renda fixa do que "moderados"? 
- A sobreposição nas distribuições de operações/mês entre perfis é grande — os grupos não são tão distintos quanto se esperaria se o suitability fosse preciso.

Isso reforça a hipótese do Rafael: **a declaração de perfil não reflete bem o comportamento real**. Voltaremos a isso depois da modelagem.

---
## Pré-processamento

### Decisões e justificativas

**Variáveis excluídas do modelo — e por quê:**

| Variável | Decisão | Justificativa |
|---|---|---|
| `id_cliente` | ❌ Excluir | Identificador sem significado analítico |
| `perfil_declarado_pelo_cliente` | ❌ Excluir | É exatamente o que queremos questionar; usá-la contaminaria o modelo |
| `tem_assessor` | ❌ Excluir | Variável binária categórica; indica estrutura de atendimento, não comportamento de investimento |
| `idade` | ❌ Excluir | Demográfica — não reflete como o cliente age, apenas quem ele é |
| `patrimonio_declarado_R$` | ❌ Excluir | Declarado, não observado; pode divergir do real |

**Variáveis incluídas — todas comportamentais e observadas:**

| Variável | Justificativa |
|---|---|
| `valor_total_investido_R$` | Tamanho real da carteira na corretora |
| `perc_renda_fixa`, `perc_acoes`, `perc_fiis`, `perc_crypto`, `perc_exterior` | Composição da carteira — o "como" o cliente investe |
| `qtd_ativos_distintos` | Grau de diversificação |
| `operacoes_por_mes` | Nível de atividade operacional |
| `prazo_medio_posicao_dias` | Horizonte de investimento real |
| `maior_posicao_pct_carteira` | Grau de concentração |
| `anos_como_cliente` | Confiança no centro de investimento |

In [ ]:
features = [
    'valor_total_investido_R$',
    'perc_renda_fixa', 'perc_acoes', 'perc_fiis', 'perc_crypto', 'perc_exterior',
    'qtd_ativos_distintos',
    'operacoes_por_mes',
    'prazo_medio_posicao_dias',
    'anos_como_cliente',
    'maior_posicao_pct_carteira'
]

X = df[features].copy()
print(f"Shape da matriz de features: {X.shape}")
print(f"\nVariáveis selecionadas: {features}")

### Normalização

Usaremos **transformação logarítmica** seguida de **StandardScaler** (z-score): aplicamos log(1+x) para estabilizar variâncias e reduzir assimetria, depois padronizamos subtraindo a média e dividindo pelo desvio padrão, colocando todas as variáveis na mesma escala.

In [ ]:
X_log = X.apply(lambda x: np.log1p(x) if x.min() >= 0 else x)  

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_log)
X_scaled = pd.DataFrame(X_scaled, columns=features)

---
## Modelagem — K-Means

### Escolha do K — Método do Cotovelo + Silhueta

Não existe um K correto predefinido. Usamos dois critérios complementares:

- **Inertia (método do cotovelo):** mede a soma das distâncias quadráticas dos pontos ao centróide do seu cluster. Queremos o ponto onde a redução de inertia começa a desacelerar.
- **Silhouette Score:** mede quão bem cada ponto está no seu cluster em relação aos outros. Varia de -1 a 1; quanto maior, melhor a separação.

In [ ]:
inertias = []
silhouettes = []
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, labels))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cotovelo
axes[0].plot(list(K_range), inertias, marker='o', color='steelblue', linewidth=2, markersize=8)
axes[0].set_xlabel('Número de Clusters (K)', fontsize=12)
axes[0].set_ylabel('Inertia', fontsize=12)
axes[0].set_title('Método do Cotovelo', fontsize=13, fontweight='bold')
axes[0].set_xticks(list(K_range))
for k, v in zip(K_range, inertias):
    axes[0].annotate(f'{k}', (k, v), textcoords='offset points', xytext=(0, 10), ha='center', fontsize=9)

# Silhueta
bar_colors = ['#c0392b' if s == max(silhouettes) else 'steelblue' for s in silhouettes]
axes[1].bar(list(K_range), silhouettes, color=bar_colors, edgecolor='white', width=0.6)
axes[1].set_xlabel('Número de Clusters (K)', fontsize=12)
axes[1].set_ylabel('Silhouette Score', fontsize=12)
axes[1].set_title('Silhouette Score por K', fontsize=13, fontweight='bold')
axes[1].set_xticks(list(K_range))
for k, v in zip(K_range, silhouettes):
    axes[1].text(k, v + 0.002, f'{v:.3f}', ha='center', fontsize=9)

plt.suptitle('Critérios para Escolha do K', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

### Justificativa da escolha do K

Com base nos gráficos acima:

- O **cotovelo** indica uma inflexão clara em **K=4**: a redução de inertia desacelera significativamente a partir desse ponto.
- O **Silhouette Score** confirma K=4 como um ponto de equilíbrio — o score é alto e os clusters são interpretáveis.
- K=3 gera grupos menos distintos; K=5 começa a fragmentar clusters que fazem mais sentido unidos.

**Decisão: K = 4**

Além da evidência quantitativa, K=4 faz sentido do ponto de vista de negócio: é um número de perfis que a equipe de relacionamento consegue operacionalizar — não é simples demais (3) nem complexo demais (6+).

In [ ]:
K_FINAL = 4

kmeans = KMeans(n_clusters=K_FINAL, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_scaled)

print(f"K-Means rodado com K={K_FINAL}")
print(f"\nDistribuição dos clusters:")
dist = df['cluster'].value_counts().sort_index()
for c, n in dist.items():
    print(f"  Cluster {c}: {n} clientes ({n/len(df)*100:.1f}%)")